# Data Preparation Workflow
**Unit 3 · Lesson 2 | PV Time Series**

Prepare hourly PV observations for next-hour regression and classification.

**Workflow:** Problem Definition → Initial Data Inspection (Structure & Schema) → Deterministic Corrections → Data Splitting.

EDA, statistical cleaning, transformations, feature engineering, and modeling are outside this lesson.

## Setup
Keep datasets outside the chapter folders so later lessons can reuse them:

```text
project_root/
├── data/
│   ├── raw/
│   │   └── PV_Data.xlsx
│   └── processed/
│       └── pv_time_series/
└── chapter_03/
    └── 03_02_data_preparation_workflow_pv_time_series.ipynb
```

Locally, start the notebook from the chapter folder; `PROJECT_ROOT = Path("..").resolve()` points to the project root. Adjust this value if your working directory differs.

In Colab, uncomment the alternative `PROJECT_ROOT` assignment below. This creates the same shared data layout under `/content/pv_course`. Load the source with one of the optional examples, then copy prepared data to Drive using Section 4.3.

Installation commands and optional imports are collected below and commented out. Uncomment only those required. Replace example paths or file IDs before running optional cells. Explanations appear in Markdown.

In [1]:
# %pip install pandas numpy openpyxl scikit-learn
# %pip install gdown

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# import shutil
# import gdown
# from google.colab import drive


In [7]:
PROJECT_ROOT = Path("..").resolve()
# PROJECT_ROOT = Path("/content/pv_course")

DATA_PATH = PROJECT_ROOT / "data/raw/PV_Data.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/pv_time_series"

### Optional: Load from Google Drive in Colab
Choose **one** of the following loading options before reading the Excel file. Neither option changes the output folder; an optional Drive backup is provided in Section 4.3.

**Option A — Mounted Drive.** Uncomment the `drive` import above and the cell below. Mount Drive, then replace the example path with your file's location under `MyDrive`. This method supports private files available in your Drive.

In [8]:
# drive.mount("/content/drive")
# DATA_PATH = Path("/content/drive/MyDrive/pv_course/data/raw/PV_Data.xlsx")

**Option B — Download by file ID.** Uncomment the `gdown` installation and import above, then the cell below. Replace `YOUR_GOOGLE_DRIVE_FILE_ID` with the ID from a sharing link:

`https://drive.google.com/file/d/FILE_ID/view`

The file must be accessible through link sharing; this does not bypass Drive permissions. Use Option A for private files. Mounting Drive is not required for this download.

The download is stored in the current runtime. In Colab it may be lost when the runtime is deleted or reset. Before publishing, restore the placeholder and clear outputs so your personal file ID is not included.

In [9]:
# file_id = "YOUR_GOOGLE_DRIVE_FILE_ID"
# DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
# gdown.download(id=file_id, output=str(DATA_PATH), quiet=True)

## 1. Problem Definition

| Item | Definition |
|---|---|
| Observation | One hourly PV-system record |
| Prediction origin | End of interval t |
| Inputs | Information available by the end of t |
| Regression target | Total PV output at t+1 |
| Classification target | Low, Medium, or High output at t+1 |
| Evaluation | Earlier observations for training; later observations for evaluation |

**Assumption:** Measurements at t are available before prediction. Delayed measurements must be lagged; observed values at t+1 cannot be inputs.

The source does not specify year or timezone, so verify temporal order from day/hour labels under a single-year assumption. Output units are described as “W or Wh”; confirm the unit before interpreting physical magnitudes.

## 2. Initial Data Inspection (Structure & Schema)
### 2.1 Load the Excel source
`DATA` has no header. Read column names from the second row of `DATA DESCRIPTION`, without appending its example records.

In [10]:
excel_file = pd.ExcelFile(DATA_PATH)
print("Sheets:", excel_file.sheet_names)

raw_data = pd.read_excel(excel_file, sheet_name="DATA", header=None)
description = pd.read_excel(excel_file, sheet_name="DATA DESCRIPTION", header=None)

raw_data.columns = description.iloc[1].tolist()

Sheets: ['DATA', 'DATA DESCRIPTION']


### CSV alternative
For a CSV with a header, use the following instead of the Excel-loading cell. Keep it commented when using the supplied Excel file. For a headerless CSV, provide `header=None` and verified column names.

In [11]:
# raw_data = pd.read_csv("pv_data.csv")

# raw_data = pd.read_csv("pv_data.csv", header=None, names=source_column_names)

### 2.2 Inspect structure and types
One row represents one hourly observation. Inspect schema and record identity only; leave distributions and correlations for training-only EDA.

In [12]:
print("Shape:", raw_data.shape)
raw_data.head()

Shape: (7080, 21)


,Day,Cosine of Day,Hour,Cosine of Hour,Air Temperature,Dewpoint,Wind Speed,Air Pressure,Weather Condition,Solar Radiation,...,PV-1 Power,PV-2 Cosine Value of the Solar Incidence Angle,PV-2 Power,PV-3 Cosine Value of the Solar Incidence Angle,PV-3 Power,PV-4 Cosine Value of the Solar Incidence Angle,PV-4 Power,PV-5 Cosine Value of the Solar Incidence Angle,PV-5 Power,Total PV Power
0,1,0.999852,1,1.0000,2,81,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
1,1,0.999852,2,0.9659,-1,86,0,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
2,1,0.999852,3,0.8660,0,87,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
3,1,0.999852,4,0.7071,-1,86,2,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0
4,1,0.999852,5,0.5000,-1,86,6,900.2,10,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0


In [13]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7080 entries, 0 to 7079
Data columns (total 21 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Day                                             7080 non-null   int64  
 1   Cosine of Day                                   7080 non-null   float64
 2   Hour                                            7080 non-null   int64  
 3   Cosine of Hour                                  7080 non-null   float64
 4   Air Temperature                                 7080 non-null   int64  
 5   Dewpoint                                        7080 non-null   int64  
 6   Wind Speed                                      7080 non-null   int64  
 7   Air Pressure                                    7080 non-null   float64
 8   Weather Condition                               7080 non-null   int64  
 9   Solar Radiation                          

In [14]:
print("Missing values:", raw_data.isna().sum().sum())

Missing values: 0


In [15]:
print("Exact duplicate rows:", raw_data.duplicated().sum())

Exact duplicate rows: 0


The supplied file contains 7,080 observations and 21 columns. Default Excel header handling would incorrectly discard the first observation. `Day` and `Hour` identify time; there is no verified calendar timestamp.

## 3. Deterministic Corrections
### 3.1 Standardize column names
Use lowercase `snake_case`. Plain string methods are sufficient for the names in this dataset; no regular-expression library is needed.

`rename` returns a renamed DataFrame while leaving `raw_data` unchanged. No separate copy or column assignment is needed.

Check that standardization has not mapped different source names to the same name. Keep the original-to-standard mapping for reference.

In [16]:
def standardize_name(name):
    name = name.strip().lower().replace("-", "_")
    return "_".join(name.split())


data = raw_data.rename(columns=standardize_name)

print(data.columns.tolist())

['day', 'cosine_of_day', 'hour', 'cosine_of_hour', 'air_temperature', 'dewpoint', 'wind_speed', 'air_pressure', 'weather_condition', 'solar_radiation', 'pv_1_cosine_value_of_the_solar_incidence_angle', 'pv_1_power', 'pv_2_cosine_value_of_the_solar_incidence_angle', 'pv_2_power', 'pv_3_cosine_value_of_the_solar_incidence_angle', 'pv_3_power', 'pv_4_cosine_value_of_the_solar_incidence_angle', 'pv_4_power', 'pv_5_cosine_value_of_the_solar_incidence_angle', 'pv_5_power', 'total_pv_power']


In [17]:
if not data.columns.is_unique:
    raise ValueError("Standardization produced duplicate column names.")

column_mapping = pd.DataFrame({
    "original_name": raw_data.columns,
    "standard_name": data.columns,
})

#### Alternative: rename specific columns
Use `rename(columns=...)` for an explicit mapping when only a few names need changing. This example starts from the original column names and leaves `data` unchanged; the workflow above standardizes all columns.

In [18]:
rename_example = raw_data.rename(columns={
    "Day": "day",
    "Hour": "hour",
    "Total PV Power": "total_pv_power",
})

print(rename_example.columns.tolist())

['day', 'Cosine of Day', 'hour', 'Cosine of Hour', 'Air Temperature', 'Dewpoint', 'Wind Speed', 'Air Pressure', 'Weather Condition', 'Solar Radiation', 'PV-1 Cosine Value of the Solar Incidence Angle', 'PV-1 Power', 'PV-2 Cosine Value of the Solar Incidence Angle', 'PV-2 Power', 'PV-3 Cosine Value of the Solar Incidence Angle', 'PV-3 Power', 'PV-4 Cosine Value of the Solar Incidence Angle', 'PV-4 Power', 'PV-5 Cosine Value of the Solar Incidence Angle', 'PV-5 Power', 'total_pv_power']


### 3.2 Review types and duplicate records
Inspect first; change only when a specific issue is confirmed. The supplied file already has numeric columns and no exact duplicate rows, so neither blanket conversion nor automatic deletion is needed.

Identical measurements can be valid at different times. Delete records only after confirming they are duplicate observations. Numeric weather codes remain categorical in meaning; encoding is deferred.

The optional lines below illustrate duplicate removal and single-column numeric conversion. Enable either only after confirming the corresponding issue.

In [19]:
duplicate_count = data.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)

# data = data.drop_duplicates().reset_index(drop=True)

# data["air_temperature"] = pd.to_numeric(data["air_temperature"], errors="raise")

Exact duplicate rows: 0


### 3.3 Validate day and hour labels
`between(1, 366)` includes **both** 1 and 366; `between(1, 24)` includes both 1 and 24. Unlike Python's `range`, the upper bound is not excluded.

Day 366 is allowed because the year is unspecified. If a non-leap year is confirmed, use 365. Hour labels follow this file's 1–24 convention, not 0–23.

`value % 1 == 0` checks that a numeric value has no fractional part. Range checks alone would incorrectly accept values such as hour 2.5.

In [20]:
valid_day = data["day"].between(1, 366) & (data["day"] % 1 == 0)
valid_hour = data["hour"].between(1, 24) & (data["hour"] % 1 == 0)

if not (valid_day & valid_hour).all():
    raise ValueError("Verify missing or invalid day/hour values in the source.")

In [21]:
if data.duplicated(subset=["day", "hour"]).any():
    raise ValueError("Repeated day/hour keys: verify the source records and year.")

chronological_order = data.sort_values(["day", "hour"]).index

if not data.index.equals(chronological_order):
    raise ValueError("Verify the source chronology before splitting.")

print("Day/hour keys are unique and chronologically ordered.")

Day/hour keys are unique and chronologically ordered.


### 3.4 Flag fixed consistency issues
Check infinite readings and negative total output without applying statistical outlier rules. Missing values were checked during inspection. Do not silently replace questionable values.

In [22]:
print("Infinite values:", np.isinf(data.to_numpy(dtype=float)).sum())
print("Negative total output:", (data["total_pv_power"] < 0).sum())

Infinite values: 0
Negative total output: 0


## 4. Data Splitting
### 4.1 Preserve chronological order
Use the earliest 70% for training, the next 15% for validation, and the latest 15% for testing.

`train_test_split` preserves the existing order when `shuffle=False`. Split twice; validation is half of the remaining 30%. Neither stratification nor a random seed is needed.

Learn data-driven decisions from training data only. Do not scale, impute, or choose thresholds using the full dataset.

In [23]:
train_data, remaining_data = train_test_split(
    data,
    test_size=0.30,
    shuffle=False,
)

validation_data, test_data = train_test_split(
    remaining_data,
    test_size=0.50,
    shuffle=False,
)

In [24]:
splits = {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
}

for name, part in splits.items():
    first = part.iloc[0]
    last = part.iloc[-1]
    print(
        f"{name}: {len(part)} rows | "
        f"day {int(first['day'])}, hour {int(first['hour'])} to "
        f"day {int(last['day'])}, hour {int(last['hour'])}"
    )

train: 4956 rows | day 1, hour 1 to day 207, hour 12
validation: 1062 rows | day 207, hour 13 to day 251, hour 18
test: 1062 rows | day 251, hour 19 to day 295, hour 24


### 4.2 Save the prepared partitions
Save the observations in chronological partitions, together with the column mapping. No target columns or time-index columns are added, and no rows are removed for target alignment.

**Later: construct model inputs and targets.** Read future output values when building samples rather than storing a separate column for each forecast horizon. The next row is not necessarily the next hour: verify temporal continuity and keep target observations within their assigned partition. Define classification thresholds using training targets only, then reuse them unchanged for validation and test.


The same export cell works locally and in Colab. It creates the output folder and any missing parent folders with `parents=True`; `exist_ok=True` allows an existing folder.

> **Colab runtime files are temporary.** Saving the notebook does not preserve these CSV files. After exporting, run the optional Drive backup in Section 4.3 before deleting or resetting the runtime.

In [25]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, part in splits.items():
    part.to_csv(OUTPUT_DIR / f"pv_time_series_{name}.csv", index=False)

column_mapping.to_csv(OUTPUT_DIR / "column_mapping.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR.resolve())

Saved outputs to: /content/drive/MyDrive/Teaching/Applied ML and DL for Photovoltaics with Python/applied-ml-dl-for-photovoltaics-with-python/data/processed/pv_time_series


### 4.3 Optional: Copy outputs to Google Drive
In Colab, uncomment the `drive` and `shutil` imports in Setup and the cell below. Run it **after exporting** the CSV files. Skip the mount line if Drive is already mounted.

Edit the destination path as needed. The cell creates the destination folder and copies the exported CSV files to persistent Drive storage without changing `OUTPUT_DIR`. Running it again overwrites files with matching names. Local users can leave this cell commented out.

This backs up the prepared partitions and column mapping, not the raw Excel file. Keep the original dataset separately in Drive if you need it across Colab sessions.

In [26]:
# drive.mount("/content/drive")
#
# drive_output_dir = Path(
#     "/content/drive/MyDrive/pv_course/data/processed/pv_time_series"
# )
# drive_output_dir.mkdir(parents=True, exist_ok=True)
#
# for csv_file in OUTPUT_DIR.glob("*.csv"):
#     shutil.copy2(csv_file, drive_output_dir / csv_file.name)
#
# print("Copied outputs to:", drive_output_dir)

## Summary
- Defined next-hour regression and classification tasks.
- Inspected the source and standardized column names.
- Checked record identity, day/hour labels, and chronological order.
- Saved a chronological 70/15/15 split without adding targets or removing observations.

**Next:** Training-only EDA. Sample construction, target alignment, classification thresholds, feature engineering, and model preprocessing are deferred to the relevant lessons.